In [1]:
import pandas as pd
import numpy as np
import os
import warnings

In [13]:
raw = "TMDB_movie_dataset_v11.csv"

columns_to_load = [
    'id', 'title', 'overview', 'poster_path', 'release_date',
    'vote_average', 'vote_count', 'runtime', 'original_language',
    'popularity', 'genres', 'budget', 'keywords', 'status'
]

movies = pd.read_csv(raw)

print(f"Dataset Loaded: {len(movies):,} rows, {len(movies.columns)} columns")
movies.head(3)


Dataset Loaded: 1,451,227 rows, 24 columns


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."


In [14]:
print("Missing Values :")
print(movies.isnull().sum())

print("\n Status s:")
print(movies['status'].value_counts())


Missing Values :
id                            0
title                        19
vote_average                  0
vote_count                    0
status                        0
release_date             330978
revenue                       0
runtime                       0
adult                         0
backdrop_path           1093710
budget                        0
homepage                1302425
imdb_id                  776182
original_language             0
original_title               19
overview                 335176
popularity                    0
poster_path              521539
tagline                 1248570
genres                   642904
production_companies     837418
production_countries     707815
spoken_languages         680279
keywords                1095388
dtype: int64

 Status s:
status
Released           1397548
In Production        24292
Post Production      16096
Planned              12068
Rumored                847
Canceled               376
Name: count, dtype: i

In [15]:
fmovies = movies.dropna(subset=['title', 'genres', 'release_date']).copy()
print(f"After dropping :  {len(fmovies):,} rows")

fmovies = fmovies[fmovies['status'] == 'Released'].copy()
print(f"After filtering : {len(fmovies):,} rows")

fmovies['release_year'] = pd.to_datetime(fmovies['release_date'], errors='coerce').dt.year
fmovies = fmovies.dropna(subset=['release_year']).copy()
fmovies['release_year'] = fmovies['release_year'].astype(int)
print(f"After year validation   : {len(fmovies):,} rows")

fmovies = fmovies.drop_duplicates(subset=['id']).copy()
print(f"After removing duplicate IDs     : {len(fmovies):,} rows")

fmovies = fmovies[(fmovies['runtime'] > 0) & (fmovies['vote_count'] >= 50)].copy()
fmovies.reset_index(drop=True, inplace=True)
print(f"After runtime > 0  vote_count >= 50: {len(fmovies):,} rows")


After dropping :  740,844 rows
After filtering : 723,781 rows
After year validation   : 723,781 rows
After removing duplicate IDs     : 723,700 rows
After runtime > 0  vote_count >= 50: 27,842 rows


In [16]:

fmovies['overview'] = fmovies['overview'].fillna('')
fmovies['keywords'] = fmovies['keywords'].fillna('')
fmovies['poster_path'] = fmovies['poster_path'].fillna('')

fmovies['content'] = fmovies['overview'] + ' ' + fmovies['keywords']

print(fmovies[['overview', 'keywords', 'poster_path', 'content']].isnull().sum())

overview       0
keywords       0
poster_path    0
content        0
dtype: int64


In [17]:
final_columns = [
    'id',
    'title',
    'overview',
    'poster_path',
    'release_date',
    'release_year',
    'vote_average',
    'vote_count',
    'runtime',
    'original_language',
    'popularity',
    'genres',
    'budget',
    'keywords',
    'content'
]

fmovies = fmovies[final_columns].copy()

print(f"Final Dataset : {fmovies.shape}")
fmovies.info()


Final Dataset : (27842, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27842 entries, 0 to 27841
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 27842 non-null  int64  
 1   title              27842 non-null  object 
 2   overview           27842 non-null  object 
 3   poster_path        27842 non-null  object 
 4   release_date       27842 non-null  object 
 5   release_year       27842 non-null  int64  
 6   vote_average       27842 non-null  float64
 7   vote_count         27842 non-null  int64  
 8   runtime            27842 non-null  int64  
 9   original_language  27842 non-null  object 
 10  popularity         27842 non-null  float64
 11  genres             27842 non-null  object 
 12  budget             27842 non-null  int64  
 13  keywords           27842 non-null  object 
 14  content            27842 non-null  object 
dtypes: float64(2), int64(5), object(8)
memory 

In [18]:
fmovies[['vote_average', 'vote_count', 'runtime', 'popularity', 'release_year', 'budget']].describe()

,vote_average,vote_count,runtime,popularity,release_year,budget
count,27842.000000,27842.000000,27842.000000,27842.000000,27842.000000,2.784200e+04
mean,6.409787,706.093133,99.750018,14.576110,2001.132677,8.986391e+06
std,0.880061,1912.199920,27.616169,40.290832,21.422578,2.552771e+07
min,1.840000,50.000000,1.000000,0.600000,1874.000000,0.000000e+00
25%,5.861000,80.000000,90.000000,6.664000,1993.000000,0.000000e+00
50%,6.475000,153.000000,98.000000,9.706500,2009.000000,0.000000e+00
75%,7.024000,444.000000,111.000000,14.760750,2016.000000,5.000000e+06
max,9.980000,34495.000000,585.000000,2994.357000,2023.000000,4.600000e+08


In [19]:
fmovies[['title', 'release_year', 'genres', 'vote_average', 'runtime', 'original_language', 'poster_path']].head(5)

,title,release_year,genres,vote_average,runtime,original_language,poster_path
0,Inception,2010,"Action, Science Fiction, Adventure",8.364,148,en,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg
1,Interstellar,2014,"Adventure, Drama, Science Fiction",8.417,169,en,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
2,The Dark Knight,2008,"Drama, Action, Crime, Thriller",8.512,152,en,/qJ2tW6WMUDux911r6m7haRef0WH.jpg
3,Avatar,2009,"Action, Adventure, Fantasy, Science Fiction",7.573,162,en,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg
4,The Avengers,2012,"Science Fiction, Action, Adventure",7.710,143,en,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg


In [20]:
final_csv_file = "final_preprocessed_movies.csv"

fmovies.to_csv(final_csv_file, index=False)

file_size_mb = os.path.getsize(final_csv_file) / (1024 * 1024)

print(f"Output File Name      : {final_csv_file}")
print(f"Total Movies Retained : {len(fmovies):,}")
print(f"Total Columns Saved   : {len(fmovies.columns)}")



Output File Name      : final_preprocessed_movies.csv
Total Movies Retained : 27,842
Total Columns Saved   : 15


In [21]:
fmovies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27842 entries, 0 to 27841
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 27842 non-null  int64  
 1   title              27842 non-null  object 
 2   overview           27842 non-null  object 
 3   poster_path        27842 non-null  object 
 4   release_date       27842 non-null  object 
 5   release_year       27842 non-null  int64  
 6   vote_average       27842 non-null  float64
 7   vote_count         27842 non-null  int64  
 8   runtime            27842 non-null  int64  
 9   original_language  27842 non-null  object 
 10  popularity         27842 non-null  float64
 11  genres             27842 non-null  object 
 12  budget             27842 non-null  int64  
 13  keywords           27842 non-null  object 
 14  content            27842 non-null  object 
dtypes: float64(2), int64(5), object(8)
memory usage: 3.2+ MB
